In [2]:
import pandas as pd
import re

# ===========================
# 1. LOAD DEDUP FILE
# ===========================
df = pd.read_excel("visuo_haptic_multimodal_non_duplicates_2026.xlsx")
print("Records loaded:", len(df))

# ===========================
# 2. NORMALIZATION
# ===========================
def norm(s):
    return re.sub(r"\s+", " ", str(s).lower()).strip()

# ===========================
# 3. KEYWORD GROUPS
# ===========================

# Strong multimodal / fusion
MULTIMODAL_STRONG = [
    "multimodal", "multi-modal", "cross-modal", "sensor fusion", "fusion"
]

# Weak multimodal
MULTIMODAL_WEAK = [
    "vision and touch", "visual tactile", "haptic perception",
    "tactile sensing", "embodied", "cross sensory"
]

# Vision terms
VISION = [
    "vision", "image", "camera", "visual"
]

# Touch terms
TOUCH = [
    "touch", "tactile", "haptic"
]

# AI/ML terms
AI = [
    "machine learning", "deep learning", "neural", "cnn", "transformer",
    "classification", "regression", "model", "algorithm"
]

# Exclusion terms
NON_ORIGINAL = [
    "review", "survey", "meta-analysis", "editorial", "commentary", "protocol"
]

IRRELEVANT = [
    "finance", "marketing", "economics", "education"
]

# ===========================
# 4. HELPER
# ===========================
def contains_any(text, keywords):
    return any(k in text for k in keywords)

# ===========================
# 5. SCORING FUNCTION
# ===========================
def score_record(title, abstract):
    # combine title + abstract
    combined = norm(title) + " " + norm(abstract)
    t = " " + combined + " "

    # Hard exclusions
    if contains_any(t, NON_ORIGINAL):
        return "Exclude", "Non-original research", 0
    if contains_any(t, IRRELEVANT):
        return "Exclude", "Irrelevant domain", 0

    score = 0
    reasons = []

    # multimodal
    if contains_any(t, MULTIMODAL_STRONG):
        score += 3
        reasons.append("multimodal-strong")
    elif contains_any(t, MULTIMODAL_WEAK):
        score += 1
        reasons.append("multimodal-weak")

    # vision
    if contains_any(t, VISION):
        score += 1
        reasons.append("vision")

    # touch
    if contains_any(t, TOUCH):
        score += 2
        reasons.append("touch")

    # AI/ML
    if contains_any(t, AI):
        score += 1
        reasons.append("AI")

    # Decision thresholds
    if score >= 4:
        return "Include", "Score≥4: " + ", ".join(reasons), score
    if score == 3:
        return "Maybe", "Score=3: " + ", ".join(reasons), score

    return "Exclude", "Low score: " + ", ".join(reasons), score

# ===========================
# 6. APPLY SCORING
# ===========================
out = df.copy()

# ensure abstract column exists
if "Abstract" not in out.columns:
    out["Abstract"] = ""

# apply scoring
out[["Decision", "Reason", "Score"]] = out.apply(
    lambda row: pd.Series(score_record(row["Title"], row["Abstract"])),
    axis=1
)

# ===========================
# 7. SUMMARY
# ===========================
print("\nDecision counts:")
print(out["Decision"].value_counts())

# ===========================
# 8. SAVE
# ===========================
out.to_excel("screened_multimodal_scored.xlsx", index=False)
print("\nSaved: screened_multimodal_scored.xlsx")

Records loaded: 219

Decision counts:
Decision
Exclude    112
Maybe       66
Include     41
Name: count, dtype: int64

Saved: screened_multimodal_scored.xlsx
